# Imports

In [6]:
import json
import math
import os
from sigfig import round as sigfig_round

# Paths

In [7]:
DEPOL_REPORT_DIR = "reports/8Q-ising-depol/"
AMPDAMP_REPORT_DIR = "reports/8Q-heisen-ampdamp"
DEPH_REPORT_DIR = "reports/8Q-heisen-deph"
VARIOUS_TMAX_REPORT_DIR = "reports/7Q-various-tmax"
OUTPUT_DIR = "reports/latex-support"

# 📄 Table: 8-qubit analysis

#### Version 1

In [8]:
# Set to False to show only the mean in the Error Rate column (no \pm std)
SHOW_ERROR_STD = True

# Set to True to append accuracy_vs_baseline.accuracy_pct in parens next to
# the error rate, e.g. "$0.190 (98.557\%)$". Mutually exclusive with
# SHOW_ERROR_STD — both together would clutter the error cell.
INCLUDE_ACCURACY = False

if SHOW_ERROR_STD and INCLUDE_ACCURACY:
    raise ValueError(
        "SHOW_ERROR_STD and INCLUDE_ACCURACY cannot both be True — "
        "showing std and accuracy together clutters the error rate cell. "
        "Choose one."
    )


def fmt_mean_std(mean, std):
    return f"${mean:.3f} \\pm {std:.3f}$"


def fmt_error_rate(mean, std, accuracy_pct=None, show_std=SHOW_ERROR_STD,
                    include_accuracy=INCLUDE_ACCURACY):
    if show_std and include_accuracy:
        # Defensive: should never happen given the module-level guard, but
        # this function may be called directly with explicit args too.
        raise ValueError("show_std and include_accuracy cannot both be True")
    if show_std:
        return f"${mean:.3f} \\pm {std:.3f}$"
    if include_accuracy:
        if accuracy_pct is None:
            raise ValueError("include_accuracy=True requires accuracy_pct")
        return f"${mean:.3f} ({accuracy_pct:.3f}\\%)$"
    return f"${mean:.3f}$"


def fmt_sci(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_overhead(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_int(value):
    return f"{int(round(value))}"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    rows_by_order = {int(r["order"]): r for r in data["rows"]}
    return data, rows_by_order


def build_noise_free_row(ad_data):
    mean = ad_data["noise_free_mean"]
    std = ad_data["noise_free_std"]
    return (
        r"\textbf{Noise-free VQE estimation} & -- & "
        f"{fmt_mean_std(mean, std)} & -- & -- & -- & -- & -- & -- \\\\"
    )


def build_unmitigated_block(deph_data, ad_data):
    return [
        r"\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}} & Dephasing & "
        f"{fmt_mean_std(deph_data['unmitigated_mean'], deph_data['unmitigated_std'])} "
        r"& -- & -- & -- & -- & -- & -- \\",
        r" & AD & "
        f"{fmt_mean_std(ad_data['unmitigated_mean'], ad_data['unmitigated_std'])} "
        r"& -- & -- & -- & -- & -- & -- \\",
    ]


def build_zne_block(order, d, a, show_error_std=SHOW_ERROR_STD,
                     include_accuracy=INCLUDE_ACCURACY):
    if show_error_std and include_accuracy:
        raise ValueError("show_error_std and include_accuracy cannot both be True")

    for field in ("depth", "gate", "overhead"):
        if not math.isclose(d[field], a[field], rel_tol=1e-9):
            raise ValueError(
                f"Order {order}: '{field}' differs between Dephasing "
                f"({d[field]}) and AD ({a[field]}) — expected identical "
                f"values for this classical quantity."
            )

    overhead_str = fmt_overhead(d["overhead"])
    depth_str = fmt_int(d["depth"])
    gate_str = fmt_int(d["gate"])

    d_acc = d["accuracy_vs_baseline"]["accuracy_pct"] if include_accuracy else None
    a_acc = a["accuracy_vs_baseline"]["accuracy_pct"] if include_accuracy else None

    return [
        r"\multirow{2}{*}{\textbf{ZNE of order " + str(order) + r"}} & "
        r"Dephasing & "
        f"{fmt_mean_std(d['zne_mean'], d['zne_std'])} & "
        f"{fmt_error_rate(d['error_rate']['error_rate'], d['error_rate']['error_rate_std'], d_acc, show_error_std, include_accuracy)} & "
        f"{fmt_sci(d['runtime'])} & "
        f"{fmt_sci(d['comp_cost'])} & "
        r"\multirow{2}{*}{" + overhead_str + r"} & "
        r"\multirow{2}{*}{" + depth_str + r"} & "
        r"\multirow{2}{*}{" + gate_str + r"} \\",
        r" & AD & "
        f"{fmt_mean_std(a['zne_mean'], a['zne_std'])} & "
        f"{fmt_error_rate(a['error_rate']['error_rate'], a['error_rate']['error_rate_std'], a_acc, show_error_std, include_accuracy)} & "
        f"{fmt_sci(a['runtime'])} & "
        f"{fmt_sci(a['comp_cost'])} & & & \\\\",
    ]


def build_table_body(deph_path, ad_path, show_error_std=SHOW_ERROR_STD,
                      include_accuracy=INCLUDE_ACCURACY):
    if show_error_std and include_accuracy:
        raise ValueError("show_error_std and include_accuracy cannot both be True")

    deph_data, deph_rows = load_export(deph_path)
    ad_data, ad_rows = load_export(ad_path)
    orders = sorted(set(deph_rows) & set(ad_rows))
    lines = [r"\hline"]
    lines.append(build_noise_free_row(ad_data))
    lines.append(r"\hline")
    lines.extend(build_unmitigated_block(deph_data, ad_data))
    for order in orders:
        lines.append(r"\hline")
        lines.extend(
            build_zne_block(order, deph_rows[order], ad_rows[order], show_error_std, include_accuracy)
        )
    lines.append(r"\hline")
    return "\n".join(lines)


if __name__ == "__main__":
    deph_path = os.path.join(DEPH_REPORT_DIR, "zne_multivar_dephasing.json")
    ad_path = os.path.join(AMPDAMP_REPORT_DIR, "zne_multivar_ad.json")
    body = build_table_body(deph_path, ad_path)
    out_path = os.path.join(OUTPUT_DIR, "zne_multivar_table_body.tex")
    with open(out_path, "w") as f:
        f.write(body + "\n")
    print(body)
    print("%" * 80)
    print(f"\nWritten to {out_path}")

\hline
\textbf{Noise-free VQE estimation} & -- & $-13.198 \pm 0.052$ & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}} & Dephasing & $-10.067 \pm 0.116$ & -- & -- & -- & -- & -- & -- \\
 & AD & $-11.592 \pm 0.131$ & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 1}} & Dephasing & $-12.289 \pm 0.087$ & $0.909 \pm 0.101$ & $1.786 \times 10^{11}$ & $3.918 \times 10^{15}$ & \multirow{2}{*}{$1.750 \times 10^{1}$} & \multirow{2}{*}{70} & \multirow{2}{*}{174} \\
 & AD & $-13.008 \pm 0.076$ & $0.190 \pm 0.092$ & $1.591 \times 10^{11}$ & $3.918 \times 10^{15}$ & & & \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 2}} & Dephasing & $-12.856 \pm 0.062$ & $0.342 \pm 0.081$ & $1.136 \times 10^{13}$ & $4.193 \times 10^{15}$ & \multirow{2}{*}{$3.514 \times 10^{2}$} & \multirow{2}{*}{236} & \multirow{2}{*}{616} \\
 & AD & $-13.134 \pm 0.055$ & $0.064 \pm 0.075$ & $1.012 \times 10^{13}$ & $4.160 \times 10^{15}$ & & & \\
\hline
\mu

#### Version 2

In [9]:
# Set to False to show only the mean in the Error Rate / Accuracy columns (no \pm std)
SHOW_ERROR_STD = False

# Set to True to add an Accuracy column right after the Error Rate column
INCLUDE_ACCURACY = True


def fmt_mean_std(mean, std):
    return f"${mean:.3f} \\pm {std:.3f}$"


def fmt_error_rate(mean, std, show_std=SHOW_ERROR_STD):
    if show_std:
        return f"${mean:.3f} \\pm {std:.3f}$"
    return f"${mean:.3f}$"


def fmt_accuracy(mean, std, show_std=SHOW_ERROR_STD):
    if show_std:
        return f"${mean:.3f} \\pm {std:.3f}$"
    return f"${mean:.3f}$"


def fmt_sci(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_overhead(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_int(value):
    return f"{int(round(value))}"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    rows_by_order = {int(r["order"]): r for r in data["rows"]}
    return data, rows_by_order


def build_noise_free_row(ad_data, include_accuracy=INCLUDE_ACCURACY):
    mean = ad_data["noise_free_mean"]
    std = ad_data["noise_free_std"]
    # Fixed trailing columns: Runtime, Comp Cost, Overhead, Depth, Gate = 5 cells,
    # regardless of whether the Accuracy column is included.
    cols = [
        r"\textbf{Noise-free VQE estimation}",
        "--",
        fmt_mean_std(mean, std),
        "--",
    ]
    if include_accuracy:
        cols.append("--")
    cols.extend(["--", "--", "--", "--", "--"])
    return " & ".join(cols) + r" \\"


def build_unmitigated_block(deph_data, ad_data, include_accuracy=INCLUDE_ACCURACY):
    deph_cols = [
        r"\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}}",
        "Dephasing",
        fmt_mean_std(deph_data["unmitigated_mean"], deph_data["unmitigated_std"]),
        "--",
    ]
    if include_accuracy:
        deph_cols.append("--")
    deph_cols.extend(["--", "--", "--", "--", "--"])

    ad_cols = [
        "",
        "AD",
        fmt_mean_std(ad_data["unmitigated_mean"], ad_data["unmitigated_std"]),
        "--",
    ]
    if include_accuracy:
        ad_cols.append("--")
    ad_cols.extend(["--", "--", "--", "--", "--"])

    return [
        " & ".join(deph_cols) + r" \\",
        " & ".join(ad_cols) + r" \\",
    ]


def build_zne_block(order, d, a, show_error_std=SHOW_ERROR_STD, include_accuracy=INCLUDE_ACCURACY):
    for field in ("depth", "gate", "overhead"):
        if not math.isclose(d[field], a[field], rel_tol=1e-9):
            raise ValueError(
                f"Order {order}: '{field}' differs between Dephasing "
                f"({d[field]}) and AD ({a[field]}) — expected identical "
                f"values for this classical quantity."
            )

    overhead_str = fmt_overhead(d["overhead"])
    depth_str = fmt_int(d["depth"])
    gate_str = fmt_int(d["gate"])

    def error_and_accuracy_cols(row_data):
        cols = [
            fmt_error_rate(
                row_data["error_rate"]["error_rate"],
                row_data["error_rate"]["error_rate_std"],
                show_error_std,
            )
        ]
        if include_accuracy:
            acc = row_data["accuracy_vs_baseline"]
            cols.append(
                fmt_accuracy(acc["accuracy_pct"], acc["accuracy_std_pct"], show_error_std)
            )
        return cols

    deph_cols = (
        [r"\multirow{2}{*}{\textbf{ZNE of order " + str(order) + r"}}", "Dephasing",
         fmt_mean_std(d["zne_mean"], d["zne_std"])]
        + error_and_accuracy_cols(d)
        + [fmt_sci(d["runtime"]), fmt_sci(d["comp_cost"]),
           r"\multirow{2}{*}{" + overhead_str + r"}",
           r"\multirow{2}{*}{" + depth_str + r"}",
           r"\multirow{2}{*}{" + gate_str + r"}"]
    )

    ad_cols = (
        ["", "AD", fmt_mean_std(a["zne_mean"], a["zne_std"])]
        + error_and_accuracy_cols(a)
        + [fmt_sci(a["runtime"]), fmt_sci(a["comp_cost"]), "", "", ""]
    )

    return [
        " & ".join(deph_cols) + r" \\",
        " & ".join(ad_cols) + r" \\",
    ]


def build_table_body(deph_path, ad_path, show_error_std=SHOW_ERROR_STD, include_accuracy=INCLUDE_ACCURACY):
    deph_data, deph_rows = load_export(deph_path)
    ad_data, ad_rows = load_export(ad_path)
    orders = sorted(set(deph_rows) & set(ad_rows))

    lines = [r"\hline"]
    lines.append(build_noise_free_row(ad_data, include_accuracy))
    lines.append(r"\hline")
    lines.extend(build_unmitigated_block(deph_data, ad_data, include_accuracy))
    for order in orders:
        lines.append(r"\hline")
        lines.extend(
            build_zne_block(order, deph_rows[order], ad_rows[order], show_error_std, include_accuracy)
        )
    lines.append(r"\hline")
    return "\n".join(lines)


if __name__ == "__main__":
    deph_path = os.path.join(DEPH_REPORT_DIR, "zne_multivar_dephasing.json")
    ad_path = os.path.join(AMPDAMP_REPORT_DIR, "zne_multivar_ad.json")
    body = build_table_body(deph_path, ad_path)
    out_path = os.path.join(OUTPUT_DIR, "zne_multivar_table_body.tex")
    with open(out_path, "w") as f:
        f.write(body + "\n")
    print(body)
    print("%" * 80)
    print(f"\nWritten to {out_path}")

\hline
\textbf{Noise-free VQE estimation} & -- & $-13.198 \pm 0.052$ & -- & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}} & Dephasing & $-10.067 \pm 0.116$ & -- & -- & -- & -- & -- & -- & -- \\
 & AD & $-11.592 \pm 0.131$ & -- & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 1}} & Dephasing & $-12.289 \pm 0.087$ & $0.909$ & $93.116$ & $1.786 \times 10^{11}$ & $3.918 \times 10^{15}$ & \multirow{2}{*}{$1.750 \times 10^{1}$} & \multirow{2}{*}{70} & \multirow{2}{*}{174} \\
 & AD & $-13.008 \pm 0.076$ & $0.190$ & $98.557$ & $1.591 \times 10^{11}$ & $3.918 \times 10^{15}$ &  &  &  \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 2}} & Dephasing & $-12.856 \pm 0.062$ & $0.342$ & $97.407$ & $1.136 \times 10^{13}$ & $4.193 \times 10^{15}$ & \multirow{2}{*}{$3.514 \times 10^{2}$} & \multirow{2}{*}{236} & \multirow{2}{*}{616} \\
 & AD & $-13.134 \pm 0.055$ & $0.064$ & $99.515$ & $1.012 \times 10^{13}$ & $4.160 \times 10^{15

### Version 3: Significant Digit Adjustment

In [ ]:
# Set to False to show only the mean in the Error Rate / Accuracy columns (no \pm std)
SHOW_ERROR_STD = False

# Set to True to add an Accuracy column right after the Error Rate column
INCLUDE_ACCURACY = True

# Number of significant figures shown for Error Rate and Accuracy columns
ERROR_ACCURACY_SIGFIGS = 3

# Number of decimal places shown for the Mean +/- Std. Dev. column
MEAN_STD_DECIMALS = 2


def fmt_sigfig(value, sigfigs=ERROR_ACCURACY_SIGFIGS):
    """Format `value` to exactly `sigfigs` significant figures (fixed-point,
    trailing zeros preserved) using the `sigfig` package (pip install sigfig)
    instead of hand-rolled rounding logic.
    """
    return sigfig_round(value, sigfigs=sigfigs, notation="standard", warn=False)


def fmt_mean_std(mean, std, decimals=MEAN_STD_DECIMALS):
    # Mean and Std. Dev. are shown to a fixed number of decimal places (the
    # std. dev., from only 10 samples, only carries 1 significant digit, so
    # the mean is limited to the same decimal precision).
    return f"${mean:.{decimals}f} \\pm {std:.{decimals}f}$"


def fmt_error_rate(mean, std, show_std=SHOW_ERROR_STD, sigfigs=ERROR_ACCURACY_SIGFIGS):
    if show_std:
        return f"${fmt_sigfig(mean, sigfigs)} \\pm {fmt_sigfig(std, sigfigs)}$"
    return f"${fmt_sigfig(mean, sigfigs)}$"


def fmt_accuracy(mean, std, show_std=SHOW_ERROR_STD, sigfigs=ERROR_ACCURACY_SIGFIGS):
    if show_std:
        return f"${fmt_sigfig(mean, sigfigs)} \\pm {fmt_sigfig(std, sigfigs)}$"
    return f"${fmt_sigfig(mean, sigfigs)}$"


def fmt_sci(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_overhead(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_int(value):
    return f"{int(round(value))}"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    rows_by_order = {int(r["order"]): r for r in data["rows"]}
    return data, rows_by_order


def build_noise_free_row(ad_data, include_accuracy=INCLUDE_ACCURACY, mean_std_decimals=MEAN_STD_DECIMALS):
    mean = ad_data["noise_free_mean"]
    std = ad_data["noise_free_std"]
    # Fixed trailing columns: Runtime, Comp Cost, Overhead, Depth, Gate = 5 cells,
    # regardless of whether the Accuracy column is included.
    cols = [
        r"\textbf{Noise-free VQE estimation}",
        "--",
        fmt_mean_std(mean, std, mean_std_decimals),
        "--",
    ]
    if include_accuracy:
        cols.append("--")
    cols.extend(["--", "--", "--", "--", "--"])
    return " & ".join(cols) + r" \\"


def build_unmitigated_block(deph_data, ad_data, include_accuracy=INCLUDE_ACCURACY,
                             mean_std_decimals=MEAN_STD_DECIMALS):
    deph_cols = [
        r"\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}}",
        "Dephasing",
        fmt_mean_std(deph_data["unmitigated_mean"], deph_data["unmitigated_std"], mean_std_decimals),
        "--",
    ]
    if include_accuracy:
        deph_cols.append("--")
    deph_cols.extend(["--", "--", "--", "--", "--"])

    ad_cols = [
        "",
        "AD",
        fmt_mean_std(ad_data["unmitigated_mean"], ad_data["unmitigated_std"], mean_std_decimals),
        "--",
    ]
    if include_accuracy:
        ad_cols.append("--")
    ad_cols.extend(["--", "--", "--", "--", "--"])

    return [
        " & ".join(deph_cols) + r" \\",
        " & ".join(ad_cols) + r" \\",
    ]


def build_zne_block(order, d, a, show_error_std=SHOW_ERROR_STD, include_accuracy=INCLUDE_ACCURACY,
                     sigfigs=ERROR_ACCURACY_SIGFIGS, mean_std_decimals=MEAN_STD_DECIMALS):
    for field in ("depth", "gate", "overhead"):
        if not math.isclose(d[field], a[field], rel_tol=1e-9):
            raise ValueError(
                f"Order {order}: '{field}' differs between Dephasing "
                f"({d[field]}) and AD ({a[field]}) — expected identical "
                f"values for this classical quantity."
            )

    overhead_str = fmt_overhead(d["overhead"])
    depth_str = fmt_int(d["depth"])
    gate_str = fmt_int(d["gate"])

    def error_and_accuracy_cols(row_data):
        cols = [
            fmt_error_rate(
                row_data["error_rate"]["error_rate"],
                row_data["error_rate"]["error_rate_std"],
                show_error_std,
                sigfigs,
            )
        ]
        if include_accuracy:
            acc = row_data["accuracy_vs_baseline"]
            cols.append(
                fmt_accuracy(
                    acc["accuracy_pct"], acc["accuracy_std_pct"], show_error_std, sigfigs
                )
            )
        return cols

    deph_cols = (
        [r"\multirow{2}{*}{\textbf{ZNE of order " + str(order) + r"}}", "Dephasing",
         fmt_mean_std(d["zne_mean"], d["zne_std"], mean_std_decimals)]
        + error_and_accuracy_cols(d)
        + [fmt_sci(d["runtime"]), fmt_sci(d["comp_cost"]),
           r"\multirow{2}{*}{" + overhead_str + r"}",
           r"\multirow{2}{*}{" + depth_str + r"}",
           r"\multirow{2}{*}{" + gate_str + r"}"]
    )

    ad_cols = (
        ["", "AD", fmt_mean_std(a["zne_mean"], a["zne_std"], mean_std_decimals)]
        + error_and_accuracy_cols(a)
        + [fmt_sci(a["runtime"]), fmt_sci(a["comp_cost"]), "", "", ""]
    )

    return [
        " & ".join(deph_cols) + r" \\",
        " & ".join(ad_cols) + r" \\",
    ]


def build_table_body(deph_path, ad_path, show_error_std=SHOW_ERROR_STD, include_accuracy=INCLUDE_ACCURACY,
                      sigfigs=ERROR_ACCURACY_SIGFIGS, mean_std_decimals=MEAN_STD_DECIMALS):
    deph_data, deph_rows = load_export(deph_path)
    ad_data, ad_rows = load_export(ad_path)
    orders = sorted(set(deph_rows) & set(ad_rows))

    lines = [r"\hline"]
    lines.append(build_noise_free_row(ad_data, include_accuracy, mean_std_decimals))
    lines.append(r"\hline")
    lines.extend(build_unmitigated_block(deph_data, ad_data, include_accuracy, mean_std_decimals))
    for order in orders:
        lines.append(r"\hline")
        lines.extend(
            build_zne_block(
                order, deph_rows[order], ad_rows[order],
                show_error_std, include_accuracy, sigfigs, mean_std_decimals,
            )
        )
    lines.append(r"\hline")
    return "\n".join(lines)


if __name__ == "__main__":
    deph_path = os.path.join(DEPH_REPORT_DIR, "zne_multivar_dephasing.json")
    ad_path = os.path.join(AMPDAMP_REPORT_DIR, "zne_multivar_ad.json")
    body = build_table_body(deph_path, ad_path)
    out_path = os.path.join(OUTPUT_DIR, "zne_multivar_table_body.tex")
    with open(out_path, "w") as f:
        f.write(body + "\n")
    print(body)
    print("%" * 80)
    print(f"\nWritten to {out_path}")

\hline
\textbf{Noise-free VQE estimation} & -- & $-13.20 \pm 0.05$ & -- & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}} & Dephasing & $-10.07 \pm 0.12$ & -- & -- & -- & -- & -- & -- & -- \\
 & AD & $-11.59 \pm 0.13$ & -- & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 1}} & Dephasing & $-12.29 \pm 0.09$ & $0.9085$ & $93.12$ & $1.786 \times 10^{11}$ & $3.918 \times 10^{15}$ & \multirow{2}{*}{$1.750 \times 10^{1}$} & \multirow{2}{*}{70} & \multirow{2}{*}{174} \\
 & AD & $-13.01 \pm 0.08$ & $0.1904$ & $98.56$ & $1.591 \times 10^{11}$ & $3.918 \times 10^{15}$ &  &  &  \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 2}} & Dephasing & $-12.86 \pm 0.06$ & $0.3422$ & $97.41$ & $1.136 \times 10^{13}$ & $4.193 \times 10^{15}$ & \multirow{2}{*}{$3.514 \times 10^{2}$} & \multirow{2}{*}{236} & \multirow{2}{*}{616} \\
 & AD & $-13.13 \pm 0.05$ & $0.06407$ & $99.51$ & $1.012 \times 10^{13}$ & $4.160 \times 10^{15}$ &  &  &  \

# 📄 Table: 8-qubit, local depolarizing

In [11]:
# Number of decimal places shown for the Mean +/- Std. Dev. column.
# (With only 10 sample runs, the std. dev. itself only carries 1 significant
# digit, so the mean is limited to the same decimal precision.)
MEAN_STD_DECIMALS = 2

# Number of digits after the decimal point in the mantissa of the ZNE
# Sampling Overhead column, e.g. precision=2 -> "1.75 \times 10^{1}".
OVERHEAD_PRECISION = 2


def fmt_mean_std(mean, std, decimals=MEAN_STD_DECIMALS):
    return f"${mean:.{decimals}f} \\pm {std:.{decimals}f}$"


def fmt_overhead(value, precision=OVERHEAD_PRECISION):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    rows_by_order = {int(r["order"]): r for r in data["rows"]}
    return data, rows_by_order


def build_table_body(path, mean_std_decimals=MEAN_STD_DECIMALS, overhead_precision=OVERHEAD_PRECISION):
    data, rows_by_order = load_export(path)

    lines = [r"\hline"]
    lines.append(
        " & ".join(["Noise-free estimation",
                    fmt_mean_std(data["noise_free_mean"], data["noise_free_std"], mean_std_decimals),
                    "--"]) + r" \\"
    )
    lines.append(r"\hline")
    lines.append(
        " & ".join(["Unmitigated",
                    fmt_mean_std(data["unmitigated_mean"], data["unmitigated_std"], mean_std_decimals),
                    "--"]) + r" \\"
    )
    lines.append(r"\hline")
    for order in sorted(rows_by_order):
        row = rows_by_order[order]
        lines.append(
            " & ".join([f"ZNE of order {order}",
                        fmt_mean_std(row["zne_mean"], row["zne_std"], mean_std_decimals),
                        fmt_overhead(row["overhead"], overhead_precision)]) + r" \\"
        )
    lines.append(r"\hline")
    return "\n".join(lines)


if __name__ == "__main__":
    body = build_table_body(DEPOL_REPORT_DIR + "/zne_multivar_depolarizing.json")
    out_path = os.path.join(OUTPUT_DIR, "8q_ising_depol_table.tex")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(out_path, "w") as f:
        f.write(body + "\n")
    print(body)
    print("%" * 80)
    print(f"\nWritten to {out_path}")

\hline
Noise-free estimation & $-9.66 \pm 0.05$ & -- \\
\hline
Unmitigated & $-7.03 \pm 0.16$ & -- \\
\hline
ZNE of order 1 & $-8.82 \pm 0.12$ & $1.75 \times 10^{1}$ \\
ZNE of order 2 & $-9.37 \pm 0.09$ & $3.51 \times 10^{2}$ \\
ZNE of order 3 & $-9.52 \pm 0.08$ & $4.35 \times 10^{3}$ \\
ZNE of order 4 & $-9.54 \pm 0.07$ & $3.87 \times 10^{4}$ \\
ZNE of order 5 & $-9.51 \pm 0.07$ & $3.15 \times 10^{5}$ \\
\hline
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Written to reports/latex-support/8q_ising_depol_table.tex


# 📄 Table: 7-qubit univariate ZNE

In [14]:
REPORT_PATH = f"{VARIOUS_TMAX_REPORT_DIR}/t20_univariate_zne.json"


# Number of significant figures for the mean in the Estimated Value column.
# The std. dev. is shown with the same number of decimal places as the
# rounded mean (so the pair lines up), rather than being sig-fig'd itself.
MEAN_STD_SIGFIGS = 3

# Number of decimal places shown for the Var(E_Ric(0)) Scaling column.
VAR_SCALING_DECIMALS = 2


def fmt_mean_std(mean, std, sigfigs=MEAN_STD_SIGFIGS):
    mean_str = sigfig_round(mean, sigfigs=sigfigs, notation="standard", warn=False)
    decimals = len(mean_str.split(".")[1]) if "." in mean_str else 0
    return f"${mean_str} \\pm {std:.{decimals}f} $"


def fmt_var_scaling(value, decimals=VAR_SCALING_DECIMALS):
    return f"${value:.{decimals}f}$"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    # Each entry is keyed by e.g. "ric1".."ric6"; sort by its "order" field.
    entries_by_order = {entry["order"]: entry for entry in data.values()}
    return entries_by_order


def build_table_body(path, mean_std_sigfigs=MEAN_STD_SIGFIGS, var_scaling_decimals=VAR_SCALING_DECIMALS):
    entries_by_order = load_export(path)
    orders = sorted(entries_by_order)

    # The entry with the highest order has the superset of base-noise sample
    # points (sorted_noise / mean_exp_vals / std_exp_vals), since each higher
    # ZNE order includes all noise points used by lower orders plus more.
    max_order_entry = entries_by_order[orders[-1]]
    noise_levels = max_order_entry["sorted_noise"]
    base_means = max_order_entry["mean_exp_vals"]
    base_stds = max_order_entry["std_exp_vals"]

    lines = [r"\hline"]

    # Noise-free estimation row: taken from mean_noise_off / std_noise_off,
    # which is constant across all order entries.
    lines.append(
        " & ".join([
            "Noise-free estimation",
            fmt_mean_std(max_order_entry["mean_noise_off"], max_order_entry["std_noise_off"], mean_std_sigfigs),
            "--",
        ]) + r" \\"
    )
    lines.append(r"\hline")

    # Base-noise rows: "At base noise Lambda^(1)=..." then "At Lambda^(k)=..."
    for i, (lam, mean, std) in enumerate(zip(noise_levels, base_means, base_stds), start=1):
        label = f"At base noise $\\Lambda^{{({i})}}= {lam}$" if i == 1 else f"At $\\Lambda^{{({i})}}= {lam}$"
        lines.append(
            " & ".join([label, fmt_mean_std(mean, std, mean_std_sigfigs), "--"]) + r" \\"
        )
    lines.append(r"\hline")

    # ZNE order rows.
    for order in orders:
        entry = entries_by_order[order]
        lines.append(
            " & ".join([
                f"ZNE of order {order}",
                fmt_mean_std(entry["zne_mean"], entry["zne_std"], mean_std_sigfigs),
                fmt_var_scaling(entry["zne_cost_mean"], var_scaling_decimals),
            ]) + r" \\"
        )
    lines.append(r"\hline")

    return "\n".join(lines)


if __name__ == "__main__":
    body = build_table_body(REPORT_PATH)
    out_path = os.path.join(OUTPUT_DIR, "tmax20_table_body.tex")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(out_path, "w") as f:
        f.write(body + "\n")
    print(body)
    print("%" * 80)
    print(f"\nWritten to {out_path}")

\hline
Noise-free estimation & $-8.41 \pm 0.07 $ & -- \\
\hline
At base noise $\Lambda^{(1)}= 1$ & $-7.66 \pm 0.10 $ & -- \\
At $\Lambda^{(2)}= 3$ & $-6.29 \pm 0.15 $ & -- \\
At $\Lambda^{(3)}= 5$ & $-5.18 \pm 0.19 $ & -- \\
At $\Lambda^{(4)}= 7$ & $-4.28 \pm 0.21 $ & -- \\
At $\Lambda^{(5)}= 9$ & $-3.54 \pm 0.22 $ & -- \\
At $\Lambda^{(6)}= 11$ & $-2.94 \pm 0.22 $ & -- \\
At $\Lambda^{(7)}= 13$ & $-2.45 \pm 0.21 $ & -- \\
\hline
ZNE of order 1 & $-8.34 \pm 0.07 $ & $2.50$ \\
ZNE of order 2 & $-8.44 \pm 0.06 $ & $5.22$ \\
ZNE of order 3 & $-8.46 \pm 0.06 $ & $11.39$ \\
ZNE of order 4 & $-8.46 \pm 0.06 $ & $27.60$ \\
ZNE of order 5 & $-8.46 \pm 0.06 $ & $74.27$ \\
ZNE of order 6 & $-8.46 \pm 0.06 $ & $217.11$ \\
\hline
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Written to reports/latex-support/tmax20_table_body.tex
